In [15]:
import pandas as pd
import numpy as np
import streamlit as st
import pydeck as pdk
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import pickle

In [16]:
data_url = 'headbrain.xlsx'

st.title("Head Size Detection")
st.markdown("This streamlit web application is a dashboard for detecting human head sizes 🗣")

#image = Image.open('pic.gif')
st.image('head.gif', width='50%', caption='Human Head', use_column_width=True)

2024-09-26 22:31:35.515 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:31:35.672 
  command:

    streamlit run /home/mercy/.local/lib/python3.9/site-packages/ipykernel_launcher.py [ARGUMENTS]
2024-09-26 22:31:35.673 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:31:35.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:31:35.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [19]:
def load_data():
    data = pd.read_excel(data_url)
    lowercase = lambda x: str(x).lower()
    data.rename(lowercase, axis='columns', inplace=True)
    data.rename(columns={'age range': 'age_range', 'head size(cm^3)': 'head_size', 'head size': 'head_size', 'brain weight(grams)': 'brain_weight'}, inplace=True)
    return data

data = load_data()

In [20]:
if st.checkbox('Show dataset', True):
    st.subheader("Dataset")
    st.write(data)


2024-09-26 22:32:46.549 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.551 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.553 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.553 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.554 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:32:46.632 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [21]:
st.subheader("Size of the dataset")
st.write('data_shape', data.shape)

2024-09-26 22:33:00.023 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:00.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:00.026 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:00.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:00.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:00.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [22]:
st.subheader("Breakdown of head size and weight by gender")
fig = px.violin(data, x='gender', y='brain_weight', height=500, width=900, points="all", box=True, color='gender', title='Violin plot with boxes showing breakdown of brain weight within different gender')
newnames = {'1': 'Male', '2': 'Female'}
fig.for_each_trace(lambda t: t.update(name=newnames[t.name],
                                      legendgroup=newnames[t.name],
                                      hovertemplate=t.hovertemplate.replace(t.name, newnames[t.name])))
st.write(fig)

2024-09-26 22:33:15.532 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:15.535 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:15.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:15.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:15.845 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [23]:
st.subheader("Relationship between brain weight and head size")
fig_1 = px.scatter(data, x='head_size', y='brain_weight', color='age_range', title='Scatter plot showing relationship between brain weight and head size by the age range')
st.write(fig_1.update_traces(showlegend=False))

2024-09-26 22:33:27.689 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:27.691 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:27.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:27.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:27.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [24]:
fig_2 = px.scatter(data, x='head_size', y='brain_weight', color='gender', title='Scatter plot showing relationship between brain weight and head size according to the gender')
st.write(fig_2.update_traces(showlegend=False))

2024-09-26 22:33:45.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:45.708 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:45.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [25]:
st.subheader("Random Forest Regressor")
X = data[['head_size', 'age_range', 'gender']]
y = data['brain_weight'].values

2024-09-26 22:33:56.496 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:33:56.498 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

reg = RandomForestRegressor(n_estimators=100, random_state=1)
reg.fit(X_train, y_train)
y_train_pred = reg.predict(X_train)
r2_train_score = r2_score(y_train, y_train_pred)

st.write("R-Squared using Random Forest Regressor: ", r2_train_score)

2024-09-26 22:34:09.460 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:34:09.461 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:34:09.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:34:09.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [27]:
st.write("The model is built, now the model is pickled so that it can be used in the future")
pickle.dump(reg, open('random_forest.pkl', 'wb'))

st.subheader("APP")

pickle_a = open("random_forest.pkl", "rb")
regressor = pickle.load(pickle_a)  # our model

2024-09-26 22:36:49.566 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:36:49.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:36:49.568 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:36:49.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:36:49.577 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:36:49.579 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [28]:
def predict_value(headsize, agerange, gender):
    prediction = regressor.predict([[headsize, agerange, gender]])  # predictions using our model
    return prediction

In [29]:
def main():
    st.title("Head weight prediction APP using ML")
    html_temp = """
        <div>
        <h2>Head Weight Prediction ML App</h2>
        </div>
        """
    st.markdown(html_temp, unsafe_allow_html=True)
    headsize = st.slider("head_size", 2500, 5000)
    agerange = st.slider("age_range", 1, 2)
    gender = st.slider("gender", 1, 2)
    result = ""
    if st.button("Predict"):
        result = predict_value(headsize, agerange, gender)
    st.success("The head weight of that person is: {} grams".format(result))

if __name__ == '__main__':
    main()

2024-09-26 22:37:17.510 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.511 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.512 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.513 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-09-26 22:37:17.517 Session state does not function when running a script without `streamlit run`
2024-09-26 22:37